<a href="https://colab.research.google.com/github/agaba123/masters_project/blob/main/data_set_masters_thesis(FINAL).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ERA5 Climate Processing Pipeline (Test on 1 Year)
# Rainfall + Temperature → Weekly climate dataset → Season model
# ============================================================

# Import required libraries
import xarray as xr
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# STEP 1: Load ERA5 NetCDF files
# ------------------------------------------------------------
# Rainfall file (tp = total precipitation)
rain_file = "data_stream-oper_stepType-accum.nc"

# Temperature file (t2m = 2 meter air temperature)
temp_file = "data_stream-oper_stepType-instant.nc"

print("Loading ERA5 files...")

rain_ds = xr.open_dataset(rain_file)
temp_ds = xr.open_dataset(temp_file)

print("Rain dataset variables:", list(rain_ds.data_vars))
print("Temp dataset variables:", list(temp_ds.data_vars))

# ------------------------------------------------------------
# STEP 2: Merge rainfall and temperature datasets
# ------------------------------------------------------------
# This combines both variables into a single dataset

ds = xr.merge([rain_ds, temp_ds])

print("\nMerged Dataset Structure")
print(ds)

# ------------------------------------------------------------
# STEP 3: Convert to Pandas DataFrame
# ------------------------------------------------------------
# Flatten the spatial grid into tabular format

df = ds[["tp", "t2m"]].to_dataframe().reset_index()

print("\nRaw dataset size:", df.shape)

# ------------------------------------------------------------
# STEP 4: Clean dataset
# ------------------------------------------------------------

# Rename ERA5 time coordinate
df = df.rename(columns={"valid_time": "time"})

# Convert temperature from Kelvin → Celsius
df["t2m"] = df["t2m"] - 273.15

# Convert rainfall from meters → millimeters
df["tp"] = df["tp"] * 1000

# Remove missing values
df = df.dropna()

print("Clean dataset size:", df.shape)

# ------------------------------------------------------------
# STEP 5: Aggregate to Weekly Climate Data
# ------------------------------------------------------------
# IMPORTANT:
# Rainfall should be SUMMED
# Temperature should be AVERAGED

weekly = (
    df.groupby(["latitude", "longitude", pd.Grouper(key="time", freq="W")])
      .agg({
          "tp": "sum",    # Weekly rainfall total
          "t2m": "mean"   # Weekly average temperature
      })
      .reset_index()
)

print("\nWeekly dataset size:", weekly.shape)

# ------------------------------------------------------------
# STEP 6: Inspect Rainfall Distribution
# ------------------------------------------------------------

print("\nRainfall statistics (weekly mm)")
print(weekly["tp"].describe())

# ------------------------------------------------------------
# STEP 7: Classify Climate Seasons
# ------------------------------------------------------------
# Simple agro-climate thresholds suitable for Western Uganda

def classify_season(rain):

    # Dry season
    if rain < 5:
        return 0

    # Transition season
    elif rain < 25:
        return 1

    # Rainy season
    else:
        return 2


weekly["season"] = weekly["tp"].apply(classify_season)

print("\nSeason distribution")
print(weekly["season"].value_counts())

# ------------------------------------------------------------
# STEP 8: Prepare Machine Learning Dataset
# ------------------------------------------------------------

X = weekly[["tp", "t2m"]]
y = weekly["season"]

# Split training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ------------------------------------------------------------
# STEP 9: Train Climate Model
# ------------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# ------------------------------------------------------------
# STEP 10: Evaluate Model
# ------------------------------------------------------------

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("\nModel accuracy:", accuracy)

# ------------------------------------------------------------
# STEP 11: Save Processed Dataset (optional)
# ------------------------------------------------------------

weekly.to_csv("weekly_climate_dataset.csv", index=False)

print("\nWeekly dataset saved: weekly_climate_dataset.csv")

Loading ERA5 files...
Rain dataset variables: ['tp']
Temp dataset variables: ['t2m']

Merged Dataset Structure
<xarray.Dataset> Size: 13MB
Dimensions:     (valid_time: 8784, latitude: 17, longitude: 11)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 1980-01-01 ... 1980-12-31T23...
  * latitude    (latitude) float64 136B 2.5 2.25 2.0 1.75 ... -1.0 -1.25 -1.5
  * longitude   (longitude) float64 88B 29.0 29.25 29.5 ... 31.0 31.25 31.5
    number      int64 8B 0
    expver      (valid_time) <U4 141kB '0001' '0001' '0001' ... '0001' '0001'
Data variables:
    tp          (valid_time, latitude, longitude) float32 7MB ...
    t2m         (valid_time, latitude, longitude) float32 7MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:           

/tmp/ipykernel_719/4084558165.py:37: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge([rain_ds, temp_ds])
/tmp/ipykernel_719/4084558165.py:37: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge([rain_ds, temp_ds])



Raw dataset size: (1642608, 7)
Clean dataset size: (1642608, 7)

Weekly dataset size: (9911, 5)

Rainfall statistics (weekly mm)
count    9911.000000
mean       40.330021
std        53.287193
min         0.000000
25%        10.382891
50%        27.594328
75%        52.270649
max       949.277649
Name: tp, dtype: float64

Season distribution
season
2    5299
1    3105
0    1507
Name: count, dtype: int64

Model accuracy: 0.9994957135653051

Weekly dataset saved: weekly_climate_dataset.csv
